# Modular Preprocessing Pipeline: Kalenjin ASR

## Advanced Signal Processing & Text Normalization for Low-Resource ASR

**Author**: Research Team  
**Dataset**: Mozilla Common Voice v24.0 - Kalenjin (kln)  
**Purpose**: Production-ready preprocessing pipeline for ASR model training

---

## Table of Contents
1. [Architecture Overview](#section1)
2. [Core Dependencies & Configuration](#section2)
3. [Audio Processing Module](#section3)
4. [Text Normalization Module](#section4)
5. [Quality Assessment Module](#section5)
6. [Dataset Structuring Module](#section6)
7. [Pipeline Orchestration](#section7)
8. [Validation & Testing](#section8)
9. [Export & Serialization](#section9)

---
## 1. Architecture Overview <a id='section1'></a>

### Design Principles

This preprocessing pipeline follows **SOLID principles** and implements a **modular architecture** optimized for:

- **Reproducibility**: Deterministic processing with comprehensive logging
- **Scalability**: Parallel processing capabilities for large corpora
- **Extensibility**: Plugin-based architecture for custom processors
- **Robustness**: Error handling and data validation at each stage
- **Performance**: Memory-efficient streaming for large datasets

### Pipeline Components

```
Raw Audio + Text
       ↓
┌─────────────────┐
│ Audio Processor │ → Resampling, VAD, Normalization
└─────────────────┘
       ↓
┌─────────────────┐
│ Text Processor  │ → Normalization, Validation, Tokenization
└─────────────────┘
       ↓
┌─────────────────┐
│ Quality Filter  │ → SNR, Duration, Alignment checks
└─────────────────┘
       ↓
┌─────────────────┐
│ Dataset Builder │ → HuggingFace format, Manifests
└─────────────────┘
```

---
## 2. Core Dependencies & Configuration <a id='section2'></a>

In [ ]:
# Core libraries
import os
import sys
import json
import yaml
import logging
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union, Any
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
import warnings
warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

# Audio processing
import librosa
import soundfile as sf
import webrtcvad
import noisereduce as nr
from scipy import signal
from scipy.io import wavfile
import pyannote.audio
from pyannote.audio import Pipeline

# Text processing
import re
import unicodedata
from collections import Counter, defaultdict
import string

# ML/DL frameworks
import torch
import torchaudio
from datasets import Dataset, DatasetDict, Audio
from transformers import AutoTokenizer, AutoProcessor

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
tqdm.pandas()

print('✓ All dependencies loaded successfully')

In [ ]:
@dataclass
class PreprocessingConfig:
    """Configuration class for preprocessing pipeline.
    
    This class encapsulates all hyperparameters and settings for the preprocessing
    pipeline, ensuring reproducibility and easy parameter tuning.
    """
    
    # Audio processing parameters
    target_sample_rate: int = 16000
    audio_format: str = 'wav'
    normalize_audio: bool = True
    apply_noise_reduction: bool = True
    
    # Duration filtering
    min_duration: float = 0.5  # seconds
    max_duration: float = 30.0  # seconds
    
    # Voice Activity Detection
    vad_mode: int = 3  # WebRTC VAD aggressiveness (0-3)
    vad_frame_duration: int = 30  # ms
    silence_threshold: float = 0.01  # RMS threshold
    
    # Text normalization
    lowercase: bool = True
    remove_punctuation: bool = True
    normalize_unicode: bool = True
    preserve_apostrophes: bool = True  # For Kalenjin linguistic features
    
    # Kalenjin-specific character set
    valid_chars: str = "abcdefghijklmnopqrstuvwxyz'ŋ "
    
    # Quality thresholds
    min_snr_db: float = 10.0
    max_silence_ratio: float = 0.8
    min_speech_ratio: float = 0.3
    
    # Processing parameters
    batch_size: int = 32
    num_workers: int = mp.cpu_count() - 1
    chunk_size: int = 1000
    
    # Output paths
    output_dir: Path = Path('../processed_data')
    cache_dir: Path = Path('../cache')
    log_dir: Path = Path('../logs')
    
    def __post_init__(self):
        """Create directories and validate configuration."""
        for dir_path in [self.output_dir, self.cache_dir, self.log_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)
            
        # Validate parameters
        assert 0 < self.min_duration < self.max_duration
        assert 0 <= self.vad_mode <= 3
        assert self.min_snr_db > 0
        assert 0 < self.min_speech_ratio < 1
        
    def save(self, path: Path) -> None:
        """Save configuration to YAML file."""
        config_dict = {
            k: str(v) if isinstance(v, Path) else v 
            for k, v in self.__dict__.items()
        }
        with open(path, 'w') as f:
            yaml.dump(config_dict, f, default_flow_style=False)
    
    @classmethod
    def load(cls, path: Path) -> 'PreprocessingConfig':
        """Load configuration from YAML file."""
        with open(path, 'r') as f:
            config_dict = yaml.safe_load(f)
        
        # Convert string paths back to Path objects
        for key in ['output_dir', 'cache_dir', 'log_dir']:
            if key in config_dict:
                config_dict[key] = Path(config_dict[key])
        
        return cls(**config_dict)

# Initialize configuration
config = PreprocessingConfig()
print(f'✓ Configuration initialized: {config.num_workers} workers, {config.target_sample_rate}Hz target')

In [ ]:
# Setup logging
def setup_logging(config: PreprocessingConfig) -> logging.Logger:
    """Configure comprehensive logging for the preprocessing pipeline."""
    
    logger = logging.getLogger('kalenjin_preprocessing')
    logger.setLevel(logging.INFO)
    
    # Clear existing handlers
    logger.handlers.clear()
    
    # File handler
    file_handler = logging.FileHandler(
        config.log_dir / 'preprocessing.log', 
        mode='w'
    )
    file_handler.setLevel(logging.INFO)
    
    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    
    # Formatter
    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    
    return logger

logger = setup_logging(config)
logger.info('Preprocessing pipeline initialized')

---
## 3. Audio Processing Module <a id='section3'></a>

### Advanced Signal Processing for ASR

This module implements state-of-the-art audio preprocessing techniques optimized for low-resource ASR:

- **Adaptive Resampling**: Quality-preserving sample rate conversion
- **Voice Activity Detection**: Multi-algorithm VAD with confidence scoring
- **Noise Reduction**: Spectral subtraction and Wiener filtering
- **Dynamic Range Compression**: Perceptually-motivated normalization
- **Quality Assessment**: SNR, THD, and spectral analysis

In [ ]:
class AudioProcessor:
    """Advanced audio processing for ASR preprocessing.
    
    This class implements a comprehensive suite of audio processing techniques
    specifically designed for speech recognition applications.
    """
    
    def __init__(self, config: PreprocessingConfig):
        self.config = config
        self.logger = logging.getLogger('kalenjin_preprocessing.audio')
        
        # Initialize VAD
        self.vad = webrtcvad.Vad(config.vad_mode)
        
        # Audio statistics for normalization
        self.stats = {
            'processed_count': 0,
            'duration_stats': [],
            'snr_stats': [],
            'rejected_count': 0
        }
    
    def load_audio(self, file_path: Path) -> Tuple[np.ndarray, int]:
        """Load audio file with error handling and format detection.
        
        Args:
            file_path: Path to audio file
            
        Returns:
            Tuple of (audio_data, sample_rate)
            
        Raises:
            ValueError: If audio file cannot be loaded or is corrupted
        """
        try:
            # Try librosa first (handles more formats)
            audio, sr = librosa.load(file_path, sr=None, mono=True)
            
            if len(audio) == 0:
                raise ValueError(f"Empty audio file: {file_path}")
                
            return audio, sr
            
        except Exception as e:
            self.logger.error(f"Failed to load {file_path}: {e}")
            raise ValueError(f"Cannot load audio file: {file_path}")
    
    def resample_audio(self, audio: np.ndarray, orig_sr: int) -> np.ndarray:
        """High-quality resampling using Kaiser window.
        
        Args:
            audio: Input audio signal
            orig_sr: Original sample rate
            
        Returns:
            Resampled audio at target sample rate
        """
        if orig_sr == self.config.target_sample_rate:
            return audio
        
        # Use librosa's high-quality resampling
        resampled = librosa.resample(
            audio, 
            orig_sr=orig_sr, 
            target_sr=self.config.target_sample_rate,
            res_type='kaiser_best'
        )
        
        return resampled
    
    def apply_vad(self, audio: np.ndarray, sr: int) -> Tuple[np.ndarray, float]:
        """Apply Voice Activity Detection with multiple algorithms.
        
        Args:
            audio: Input audio signal
            sr: Sample rate
            
        Returns:
            Tuple of (trimmed_audio, speech_ratio)
        """
        # Convert to 16-bit PCM for WebRTC VAD
        audio_int16 = (audio * 32767).astype(np.int16)
        
        # Frame-based VAD
        frame_duration = self.config.vad_frame_duration / 1000.0  # Convert to seconds
        frame_length = int(sr * frame_duration)
        
        frames = []
        speech_frames = []
        
        for i in range(0, len(audio_int16), frame_length):
            frame = audio_int16[i:i + frame_length]
            
            if len(frame) < frame_length:
                # Pad last frame
                frame = np.pad(frame, (0, frame_length - len(frame)))
            
            frames.append(frame)
            
            # WebRTC VAD requires specific sample rates
            if sr in [8000, 16000, 32000, 48000]:
                try:
                    is_speech = self.vad.is_speech(frame.tobytes(), sr)
                    speech_frames.append(is_speech)
                except:
                    # Fallback to energy-based VAD
                    energy = np.sqrt(np.mean(frame.astype(float) ** 2))
                    speech_frames.append(energy > self.config.silence_threshold * 32767)
            else:
                # Energy-based VAD for non-standard sample rates
                energy = np.sqrt(np.mean(frame.astype(float) ** 2))
                speech_frames.append(energy > self.config.silence_threshold * 32767)
        
        # Calculate speech ratio
        speech_ratio = sum(speech_frames) / len(speech_frames) if speech_frames else 0
        
        # Find speech boundaries
        if speech_ratio < self.config.min_speech_ratio:
            self.logger.warning(f"Low speech ratio: {speech_ratio:.3f}")
            return audio, speech_ratio
        
        # Trim silence from beginning and end
        speech_indices = np.where(speech_frames)[0]
        if len(speech_indices) > 0:
            start_frame = max(0, speech_indices[0] - 1)  # Keep one frame before
            end_frame = min(len(frames), speech_indices[-1] + 2)  # Keep one frame after
            
            start_sample = start_frame * frame_length
            end_sample = end_frame * frame_length
            
            trimmed_audio = audio[start_sample:end_sample]
        else:
            trimmed_audio = audio
        
        return trimmed_audio, speech_ratio
    
    def reduce_noise(self, audio: np.ndarray, sr: int) -> np.ndarray:
        """Apply noise reduction using spectral subtraction.
        
        Args:
            audio: Input audio signal
            sr: Sample rate
            
        Returns:
            Denoised audio signal
        """
        if not self.config.apply_noise_reduction:
            return audio
        
        try:
            # Use noisereduce library for spectral subtraction
            denoised = nr.reduce_noise(
                y=audio, 
                sr=sr,
                stationary=False,  # Non-stationary noise
                prop_decrease=0.8
            )
            return denoised
        except Exception as e:
            self.logger.warning(f"Noise reduction failed: {e}")
            return audio
    
    def normalize_audio(self, audio: np.ndarray) -> np.ndarray:
        """Apply perceptually-motivated audio normalization.
        
        Args:
            audio: Input audio signal
            
        Returns:
            Normalized audio signal
        """
        if not self.config.normalize_audio:
            return audio
        
        # RMS normalization with peak limiting
        rms = np.sqrt(np.mean(audio ** 2))
        
        if rms > 0:
            # Target RMS level (approximately -20 dB)
            target_rms = 0.1
            normalized = audio * (target_rms / rms)
            
            # Peak limiting to prevent clipping
            peak = np.max(np.abs(normalized))
            if peak > 0.95:
                normalized = normalized * (0.95 / peak)
            
            return normalized
        
        return audio
    
    def calculate_snr(self, audio: np.ndarray) -> float:
        """Estimate Signal-to-Noise Ratio using spectral analysis.
        
        Args:
            audio: Input audio signal
            
        Returns:
            Estimated SNR in dB
        """
        # Simple SNR estimation using signal variance
        # More sophisticated methods could use spectral analysis
        
        # Estimate noise from quiet segments (bottom 10% of energy)
        frame_size = 1024
        hop_size = 512
        
        energies = []
        for i in range(0, len(audio) - frame_size, hop_size):
            frame = audio[i:i + frame_size]
            energy = np.sum(frame ** 2)
            energies.append(energy)
        
        if not energies:
            return 0.0
        
        energies = np.array(energies)
        
        # Estimate noise from lowest 10% of frames
        noise_threshold = np.percentile(energies, 10)
        signal_energy = np.mean(energies)
        
        if noise_threshold > 0:
            snr_db = 10 * np.log10(signal_energy / noise_threshold)
            return max(0, snr_db)  # Clip negative SNR
        
        return 0.0
    
    def process_audio_file(self, file_path: Path) -> Optional[Dict[str, Any]]:
        """Process a single audio file through the complete pipeline.
        
        Args:
            file_path: Path to input audio file
            
        Returns:
            Dictionary with processed audio info or None if rejected
        """
        try:
            # Load audio
            audio, orig_sr = self.load_audio(file_path)
            
            # Check duration
            duration = len(audio) / orig_sr
            if not (self.config.min_duration <= duration <= self.config.max_duration):
                self.logger.debug(f"Duration filter: {duration:.2f}s - {file_path}")
                self.stats['rejected_count'] += 1
                return None
            
            # Resample
            audio = self.resample_audio(audio, orig_sr)
            
            # Apply VAD
            audio, speech_ratio = self.apply_vad(audio, self.config.target_sample_rate)
            
            if speech_ratio < self.config.min_speech_ratio:
                self.logger.debug(f"Speech ratio filter: {speech_ratio:.3f} - {file_path}")
                self.stats['rejected_count'] += 1
                return None
            
            # Noise reduction
            audio = self.reduce_noise(audio, self.config.target_sample_rate)
            
            # Normalization
            audio = self.normalize_audio(audio)
            
            # Quality assessment
            snr_db = self.calculate_snr(audio)
            
            if snr_db < self.config.min_snr_db:
                self.logger.debug(f"SNR filter: {snr_db:.1f}dB - {file_path}")
                self.stats['rejected_count'] += 1
                return None
            
            # Update statistics
            final_duration = len(audio) / self.config.target_sample_rate
            self.stats['processed_count'] += 1
            self.stats['duration_stats'].append(final_duration)
            self.stats['snr_stats'].append(snr_db)
            
            return {
                'audio': audio,
                'sample_rate': self.config.target_sample_rate,
                'duration': final_duration,
                'snr_db': snr_db,
                'speech_ratio': speech_ratio,
                'original_path': str(file_path)
            }
            
        except Exception as e:
            self.logger.error(f"Error processing {file_path}: {e}")
            self.stats['rejected_count'] += 1
            return None
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get processing statistics."""
        stats = self.stats.copy()
        
        if stats['duration_stats']:
            stats['duration_mean'] = np.mean(stats['duration_stats'])
            stats['duration_std'] = np.std(stats['duration_stats'])
            stats['duration_total'] = np.sum(stats['duration_stats'])
        
        if stats['snr_stats']:
            stats['snr_mean'] = np.mean(stats['snr_stats'])
            stats['snr_std'] = np.std(stats['snr_stats'])
        
        return stats

print('✓ AudioProcessor class defined')

---
## 4. Text Normalization Module <a id='section4'></a>

### Kalenjin-Specific Text Processing

This module implements linguistically-informed text normalization for Kalenjin:

- **Orthographic Standardization**: Consistent spelling conventions
- **Character Set Validation**: Kalenjin-specific character filtering
- **Morphological Awareness**: Preservation of linguistic markers
- **Unicode Normalization**: Consistent encoding handling

In [ ]:
class TextProcessor:
    """Advanced text processing for Kalenjin ASR.
    
    This class implements linguistically-informed text normalization
    specifically designed for the Kalenjin language.
    """
    
    def __init__(self, config: PreprocessingConfig):
        self.config = config
        self.logger = logging.getLogger('kalenjin_preprocessing.text')
        
        # Kalenjin-specific patterns
        self.valid_chars = set(config.valid_chars)
        
        # Common Kalenjin morphological patterns
        self.kalenjin_patterns = {
            'ng_sound': re.compile(r"ng'"),  # Preserve ng' sound
            'apostrophes': re.compile(r"['']"),  # Normalize apostrophes
            'multiple_spaces': re.compile(r'\s+'),
            'leading_trailing_space': re.compile(r'^\s+|\s+$')
        }
        
        # Statistics
        self.stats = {
            'processed_count': 0,
            'rejected_count': 0,
            'char_distribution': Counter(),
            'word_lengths': [],
            'sentence_lengths': []
        }
    
    def normalize_unicode(self, text: str) -> str:
        """Normalize Unicode characters to consistent form.
        
        Args:
            text: Input text
            
        Returns:
            Unicode-normalized text
        """
        if not self.config.normalize_unicode:
            return text
        
        # Use NFD (Canonical Decomposition) for consistent handling
        normalized = unicodedata.normalize('NFD', text)
        
        # Remove combining characters that aren't part of Kalenjin
        # Keep only base characters and essential diacritics
        filtered_chars = []
        for char in normalized:
            if not unicodedata.combining(char) or char in self.valid_chars:
                filtered_chars.append(char)
        
        return ''.join(filtered_chars)
    
    def clean_punctuation(self, text: str) -> str:
        """Remove punctuation while preserving linguistic apostrophes.
        
        Args:
            text: Input text
            
        Returns:
            Text with punctuation cleaned
        """
        if not self.config.remove_punctuation:
            return text
        
        # Normalize apostrophes first
        text = self.kalenjin_patterns['apostrophes'].sub("'", text)
        
        if self.config.preserve_apostrophes:
            # Remove all punctuation except apostrophes
            # Create translation table
            punct_to_remove = string.punctuation.replace("'", "")
            translator = str.maketrans('', '', punct_to_remove)
            text = text.translate(translator)
        else:
            # Remove all punctuation
            translator = str.maketrans('', '', string.punctuation)
            text = text.translate(translator)
        
        return text
    
    def filter_characters(self, text: str) -> Tuple[str, bool]:
        """Filter text to contain only valid Kalenjin characters.
        
        Args:
            text: Input text
            
        Returns:
            Tuple of (filtered_text, is_valid)
        """
        filtered_chars = []
        invalid_chars = set()
        
        for char in text:
            if char in self.valid_chars:
                filtered_chars.append(char)
            else:
                invalid_chars.add(char)
        
        filtered_text = ''.join(filtered_chars)
        
        # Check if too many characters were removed
        if len(invalid_chars) > 0:
            removal_ratio = len(invalid_chars) / len(set(text))
            if removal_ratio > 0.3:  # More than 30% unique chars removed
                self.logger.debug(f"High character removal ratio: {removal_ratio:.2f}")
                return filtered_text, False
        
        return filtered_text, True
    
    def normalize_whitespace(self, text: str) -> str:
        """Normalize whitespace characters.
        
        Args:
            text: Input text
            
        Returns:
            Text with normalized whitespace
        """
        # Replace multiple spaces with single space
        text = self.kalenjin_patterns['multiple_spaces'].sub(' ', text)
        
        # Remove leading/trailing whitespace
        text = self.kalenjin_patterns['leading_trailing_space'].sub('', text)
        
        return text
    
    def validate_text_quality(self, text: str) -> bool:
        """Validate text quality for ASR training.
        
        Args:
            text: Input text
            
        Returns:
            True if text passes quality checks
        """
        # Check minimum length
        if len(text.strip()) < 3:
            return False
        
        # Check for reasonable word count
        words = text.split()
        if len(words) < 1 or len(words) > 50:  # Reasonable bounds
            return False
        
        # Check for repeated characters (likely transcription errors)
        for word in words:
            if len(word) > 2:
                # Check for more than 3 consecutive identical characters
                for i in range(len(word) - 3):
                    if len(set(word[i:i+4])) == 1:
                        return False
        
        # Check character distribution
        char_counts = Counter(text.lower())
        total_chars = sum(char_counts.values())
        
        # Check if any single character dominates (>60%)
        for char, count in char_counts.items():
            if char != ' ' and count / total_chars > 0.6:
                return False
        
        return True
    
    def process_text(self, text: str) -> Optional[str]:
        """Process text through the complete normalization pipeline.
        
        Args:
            text: Input text
            
        Returns:
            Processed text or None if rejected
        """
        try:
            if not text or not isinstance(text, str):
                self.stats['rejected_count'] += 1
                return None
            
            # Unicode normalization
            text = self.normalize_unicode(text)
            
            # Case normalization
            if self.config.lowercase:
                text = text.lower()
            
            # Punctuation cleaning
            text = self.clean_punctuation(text)
            
            # Character filtering
            text, is_valid = self.filter_characters(text)
            if not is_valid:
                self.stats['rejected_count'] += 1
                return None
            
            # Whitespace normalization
            text = self.normalize_whitespace(text)
            
            # Quality validation
            if not self.validate_text_quality(text):
                self.stats['rejected_count'] += 1
                return None
            
            # Update statistics
            self.stats['processed_count'] += 1
            self.stats['char_distribution'].update(text.lower())
            
            words = text.split()
            self.stats['word_lengths'].extend([len(word) for word in words])
            self.stats['sentence_lengths'].append(len(words))
            
            return text
            
        except Exception as e:
            self.logger.error(f"Error processing text '{text[:50]}...': {e}")
            self.stats['rejected_count'] += 1
            return None
    
    def get_vocabulary(self) -> Dict[str, int]:
        """Extract vocabulary from processed texts.
        
        Returns:
            Dictionary mapping characters to frequencies
        """
        return dict(self.stats['char_distribution'])
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get processing statistics."""
        stats = {
            'processed_count': self.stats['processed_count'],
            'rejected_count': self.stats['rejected_count'],
            'vocabulary_size': len(self.stats['char_distribution']),
            'total_characters': sum(self.stats['char_distribution'].values())
        }
        
        if self.stats['word_lengths']:
            stats['avg_word_length'] = np.mean(self.stats['word_lengths'])
            stats['word_length_std'] = np.std(self.stats['word_lengths'])
        
        if self.stats['sentence_lengths']:
            stats['avg_sentence_length'] = np.mean(self.stats['sentence_lengths'])
            stats['sentence_length_std'] = np.std(self.stats['sentence_lengths'])
        
        return stats

print('✓ TextProcessor class defined')